# Approve And Publish Directory Goblins

Run this with an admin JupyterHub/API token after a submitter requests review.

In [ ]:
import importlib
import os
import site
import subprocess
import sys
from pathlib import Path

package = (
    os.environ.get("GOBLIN_KING_REPOSITORY_NOTEBOOK_PACKAGE")
    or os.environ.get("GOBLIN_KING_NOTEBOOK_PACKAGE")
    or "git+https://github.com/tashabits/goblin-king.git"
)
print(f"Installing notebook helper from {package}")
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--disable-pip-version-check",
    "--quiet",
    "--user",
    "--force-reinstall",
    "--no-deps",
    package,
])
user_site = site.getusersitepackages()
if user_site not in sys.path:
    sys.path.insert(0, user_site)
importlib.invalidate_caches()
for module_name in list(sys.modules):
    if module_name == "goblin_king" or module_name.startswith("goblin_king."):
        del sys.modules[module_name]

import goblin_king  # noqa: E402, I001
from goblin_king.notebooks import GoblinKingNotebookClient  # noqa: E402, I001

token = os.environ.get("JUPYTERHUB_API_TOKEN") or os.environ.get("GOBLIN_KING_API_TOKEN")
if not token:
    raise RuntimeError("JUPYTERHUB_API_TOKEN or GOBLIN_KING_API_TOKEN is required")

client = GoblinKingNotebookClient(
    api_url=os.environ.get(
        "GOBLIN_KING_API_URL",
        "http://goblin-king-api.default.svc.cluster.local:8000",
    ),
    repository_url=os.environ.get("GOBLIN_KING_REPOSITORY_URL") or None,
    token=token,
    request_timeout_seconds=180,
)
print(f"Loaded goblin_king from {Path(goblin_king.__file__).resolve()}")

In [ ]:
pending = client.list_directory_entries(status="pending_review", limit=100)
[(item["entry"]["name"], item["entry"]["id"], item["entry"]["type"]) for item in pending["items"]]

In [ ]:
function_name = os.environ.get("GOBLIN_DIRECTORY_FUNCTION_NAME", "workbook.shared-hello")
service_name = os.environ.get("GOBLIN_DIRECTORY_SERVICE_NAME", "workbook.shared-long-hello")

def by_name(name):
    for item in pending["items"]:
        if item["entry"]["name"] == name:
            return item
    raise RuntimeError(f"No pending directory entry named {name}")

function_entry = by_name(function_name)
service_entry = by_name(service_name)
{
    "function": function_entry["entry"],
    "service": service_entry["entry"],
}

In [ ]:
function_approved = client.approve_directory_entry(
    function_entry["entry"]["id"],
    note="Approved from admin workbook",
    progress=True,
)
function_published = client.publish_directory_entry(
    function_entry["entry"]["id"],
    progress=True,
)
service_approved = client.approve_directory_entry(
    service_entry["entry"]["id"],
    note="Approved from admin workbook",
    progress=True,
)
service_published = client.publish_directory_entry(
    service_entry["entry"]["id"],
    progress=True,
)
{
    "function": function_published["entry"],
    "service": service_published["entry"],
}